# AutoOpsEnv Training Notebook (TRL + PyTorch)

This notebook trains an agent by interacting directly with `TicketEnv` and logs learning evidence.

In [ ]:
# Colab setup
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install torch transformers trl accelerate datasets matplotlib pandas pyyaml

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

project_root = Path('..').resolve()
if str(project_root) not in os.sys.path:
    os.sys.path.append(str(project_root))

from ticket_env import TicketEnv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
ACTIONS = [
    'analyze_logs',
    'search_knowledge_base',
    'ask_for_more_info',
    'propose_fix',
    'execute_fix',
]

def infer_fix_from_state(state):
    text = (state.get('ticket_description', '') + ' ' + state.get('logs', '')).lower()
    if '503' in text or 'connection refused' in text or 'upstream' in text:
        return 'restart_web_api_service'
    if 'oom' in text or 'memory' in text or 'killed' in text:
        return 'increase_worker_memory_limit'
    if 'auth' in text or 'missing required setting' in text or 'config' in text:
        return 'restore_auth_secret_config'
    if 'crashloopbackoff' in text or 'probe' in text or 'crash' in text:
        return 'patch_healthcheck_dependency_and_restart'
    if 'timeout' in text or 'port' in text or 'database' in text or 'security group' in text:
        return 'allow_db_port_5432_in_security_group'
    return 'restart_web_api_service'

def format_prompt(state):
    return (
        'You are an IT ops agent. Choose one action from: analyze_logs, search_knowledge_base, '
        'ask_for_more_info, propose_fix, execute_fix.\n'
        f"Ticket: {state['ticket_description']}\n"
        f"Logs: {state['logs']}\n"
        f"Context: {state['system_context']}\n"
        f"Diagnosed: {state['diagnosed']}\n"
        f"Proposed fix: {state['proposed_fix']}\n"
        'Answer with one action token only.'
    )

def extract_action(text, state):
    t = text.lower()
    for action in ACTIONS:
        if action in t:
            if action in ['propose_fix', 'execute_fix']:
                fix = state.get('proposed_fix') or infer_fix_from_state(state)
                return {'action': action, 'content': str(fix)}
            return action
    # Fallbacks keep the policy safe and valid.
    if not state.get('diagnosed'):
        return 'analyze_logs'
    if not state.get('proposed_fix'):
        return {'action': 'propose_fix', 'content': infer_fix_from_state(state)}
    return {'action': 'execute_fix', 'content': str(state.get('proposed_fix'))}

def random_policy_action(state):
    a = random.choice(ACTIONS)
    if a in ['propose_fix', 'execute_fix']:
        noisy = [
            'restart_web_api_service',
            'increase_worker_memory_limit',
            'restore_auth_secret_config',
            'patch_healthcheck_dependency_and_restart',
            'allow_db_port_5432_in_security_group',
            'disable_authentication',
            'open_all_ports_public',
        ]
        return {'action': a, 'content': random.choice(noisy)}
    return a

def run_episode(env, policy_fn):
    s = env.reset()
    done = False
    total_reward = 0.0
    while not done:
        action = policy_fn(s)
        s, r, done, _ = env.step(action)
        total_reward += r
    return total_reward

In [ ]:
# Baseline before training
env = TicketEnv(max_steps=6, seed=SEED)
baseline_rewards = [run_episode(env, random_policy_action) for _ in range(30)]
baseline_mean = float(np.mean(baseline_rewards))
print('Baseline mean reward:', round(baseline_mean, 3))

In [ ]:
# TRL PPO setup
model_name = 'sshleifer/tiny-gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name).to(device)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name).to(device)

ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=1,
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    optimize_cuda_cache=False,
    seed=SEED,
)

ppo_trainer = PPOTrainer(ppo_config, model, ref_model, tokenizer)
print('TRL PPO initialized')

In [ ]:
# Online RL training against the environment
train_env = TicketEnv(max_steps=6, seed=SEED)
num_episodes = 60
train_rewards = []
losses = []

for ep in range(num_episodes):
    state = train_env.reset()
    done = False
    episode_reward = 0.0

    while not done:
        prompt = format_prompt(state)
        query_tensor = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

        generation = model.generate(
            query_tensor,
            max_new_tokens=6,
            do_sample=True,
            top_p=0.9,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        response_tokens = generation[:, query_tensor.shape[1]:]
        response_text = tokenizer.decode(response_tokens[0], skip_special_tokens=True)

        action = extract_action(response_text, state)
        next_state, reward, done, info = train_env.step(action)

        reward_tensor = torch.tensor(float(reward), dtype=torch.float32).to(device)
        stats = ppo_trainer.step(
            [query_tensor[0]],
            [response_tokens[0]],
            [reward_tensor],
        )
        loss_value = float(stats.get('ppo/loss/total', stats.get('loss', 0.0)))
        losses.append(loss_value)

        episode_reward += reward
        state = next_state

    train_rewards.append(float(episode_reward))

    if (ep + 1) % 10 == 0:
        print(f'Episode {ep+1}/{num_episodes} | reward={episode_reward:.2f} | outcome={info.get("outcome")}')

In [ ]:
# Deterministic evaluation after training
def trained_policy_action(state):
    prompt = format_prompt(state)
    q = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    out = model.generate(
        q,
        max_new_tokens=6,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    r = out[:, q.shape[1]:]
    txt = tokenizer.decode(r[0], skip_special_tokens=True)
    return extract_action(txt, state)

eval_env = TicketEnv(max_steps=6, seed=SEED + 7)
after_rewards = [run_episode(eval_env, trained_policy_action) for _ in range(30)]
after_mean = float(np.mean(after_rewards))

print('Before training mean reward:', round(baseline_mean, 3))
print('After training mean reward: ', round(after_mean, 3))
print('Improvement:               ', round(after_mean - baseline_mean, 3))

In [ ]:
# Save learning evidence plots
plots_dir = (Path('..') / 'plots').resolve()
plots_dir.mkdir(parents=True, exist_ok=True)

reward_path = plots_dir / 'reward.png'
loss_path = plots_dir / 'loss.png'

plt.figure(figsize=(8, 4))
plt.plot(train_rewards, label='Episode Reward')
if len(train_rewards) >= 5:
    rolling = pd.Series(train_rewards).rolling(5).mean()
    plt.plot(rolling, label='Rolling Mean (5)', linewidth=2)
plt.title('Reward Curve: TicketEnv Training')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.legend()
plt.tight_layout()
plt.savefig(reward_path, dpi=150)
plt.close()

plt.figure(figsize=(8, 4))
plt.plot(losses, label='PPO Loss')
plt.title('Loss Curve: TRL PPO Updates')
plt.xlabel('Update Step')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig(loss_path, dpi=150)
plt.close()

print('Saved:', reward_path)
print('Saved:', loss_path)

In [ ]:
# Summary table
summary = pd.DataFrame({
    'metric': ['before_training_mean_reward', 'after_training_mean_reward', 'delta'],
    'value': [baseline_mean, after_mean, after_mean - baseline_mean],
})
summary